# Step 2 — Build Dataset

Maps CSV rows → absolute image paths, filters missing files, drops NaNs,
performs an 80/10/10 train-val-test split, fits a `StandardScaler` on the
7 regression targets (train only), and saves the scaler + split CSVs.

**Run from repo root:** `airsight/`

## 2.1 — Imports & path constants

In [1]:
import os
import pickle
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ── Repo-root anchor (works whether CWD is airsight/ or airsight/notebooks/) ──
REPO_ROOT = Path(os.getcwd())
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

COMBINED_DIR = (
    REPO_ROOT
    / "data" / "raw" / "archive (2)"
    / "Air Pollution Image Dataset"
    / "Air Pollution Image Dataset"
    / "Combined_Dataset"
)

CSV_PATH  = COMBINED_DIR / "IND_and_Nep_AQI_Dataset.csv"
IMG_DIR   = COMBINED_DIR / "All_img"

DATA_DIR      = REPO_ROOT / "data"
ARTIFACTS_DIR = REPO_ROOT / "artifacts"
DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COLS = ["PM2.5", "PM10", "O3", "CO", "SO2", "NO2", "AQI"]

RANDOM_STATE = 42

print(f"Repo root    : {REPO_ROOT}")
print(f"CSV          : {CSV_PATH}")
print(f"Image dir    : {IMG_DIR}")
print(f"Artifacts dir: {ARTIFACTS_DIR}")

Repo root    : C:\Users\Meraj\Desktop\PROJECTS\airsight
CSV          : C:\Users\Meraj\Desktop\PROJECTS\airsight\data\raw\archive (2)\Air Pollution Image Dataset\Air Pollution Image Dataset\Combined_Dataset\IND_and_Nep_AQI_Dataset.csv
Image dir    : C:\Users\Meraj\Desktop\PROJECTS\airsight\data\raw\archive (2)\Air Pollution Image Dataset\Air Pollution Image Dataset\Combined_Dataset\All_img
Artifacts dir: C:\Users\Meraj\Desktop\PROJECTS\airsight\artifacts


## 2.2 — Load CSV and build `image_path` column

In [2]:
df_raw = pd.read_csv(CSV_PATH)

print(f"Raw CSV shape : {df_raw.shape}")
print(f"Columns       : {df_raw.columns.tolist()}")

# Strip any accidental leading/trailing whitespace from filenames
df_raw["Filename"] = df_raw["Filename"].str.strip()

# Build the full absolute image path for each row
df_raw["image_path"] = df_raw["Filename"].apply(
    lambda fn: str(IMG_DIR / fn)
)

print("\nSample image_path values:")
print(df_raw["image_path"].head(3).to_string())

Raw CSV shape : (12240, 14)
Columns       : ['Location', 'Filename', 'Year', 'Month', 'Day', 'Hour', 'AQI', 'PM2.5', 'PM10', 'O3', 'CO', 'SO2', 'NO2', 'AQI_Class']

Sample image_path values:
0    C:\Users\Meraj\Desktop\PROJECTS\airsight\data\...
1    C:\Users\Meraj\Desktop\PROJECTS\airsight\data\...
2    C:\Users\Meraj\Desktop\PROJECTS\airsight\data\...


## 2.3 — Filter rows where the image file does not exist on disk

In [3]:
# Use os.path.exists — handles spaces and unicode in paths correctly
exists_mask = df_raw["image_path"].apply(os.path.exists)

n_missing = (~exists_mask).sum()
print(f"Rows with missing image : {n_missing}")
print(f"Rows with valid image   : {exists_mask.sum()}")

if n_missing > 0:
    print("\nSample missing paths:")
    print(df_raw.loc[~exists_mask, "image_path"].head(5).to_string())

df_filtered = df_raw[exists_mask].copy()
print(f"\nShape after filter: {df_filtered.shape}")

Rows with missing image : 0
Rows with valid image   : 12240

Shape after filter: (12240, 15)


## 2.4 — Select columns, drop NaN rows

In [4]:
KEEP_COLS = ["image_path"] + TARGET_COLS
df_clean = df_filtered[KEEP_COLS].copy()

before = len(df_clean)
df_clean.dropna(inplace=True)
after  = len(df_clean)

print(f"Rows before dropna : {before}")
print(f"Rows after  dropna : {after}  (dropped {before - after} NaN rows)")
print(f"\nTarget column dtypes:")
print(df_clean[TARGET_COLS].dtypes.to_string())
print(f"\nDescriptive stats:")
print(df_clean[TARGET_COLS].describe().round(3).to_string())

Rows before dropna : 12240
Rows after  dropna : 10610  (dropped 1630 NaN rows)

Target column dtypes:
PM2.5    float64
PM10     float64
O3       float64
CO       float64
SO2      float64
NO2      float64
AQI        int64

Descriptive stats:
           PM2.5       PM10         O3         CO        SO2        NO2        AQI
count  10610.000  10610.000  10610.000  10610.000  10610.000  10610.000  10610.000
mean     132.188    139.444     37.248    106.644     13.187     31.811    166.254
std      129.681    107.183     32.846    119.748      9.877     34.156    108.777
min        4.000      7.000      1.000      0.250      2.000      0.670     15.000
25%       33.180     61.000     10.000      3.000      4.400      6.000     91.000
50%       67.060    100.820     26.000     52.000     10.000     18.000    141.000
75%      193.000    198.000     58.890    204.000     20.000     47.000    230.000
max      500.000    480.000    225.000    410.000     48.000    119.000    450.000


## 2.5 — 80 / 10 / 10 train-val-test split

In [5]:
# First split: 80% train, 20% temp
df_train, df_temp = train_test_split(
    df_clean, test_size=0.20, random_state=RANDOM_STATE, shuffle=True
)

# Second split: 50% of temp → val, 50% → test  (10% each of total)
df_val, df_test = train_test_split(
    df_temp, test_size=0.50, random_state=RANDOM_STATE, shuffle=True
)

print(f"Split sizes (before scaling):")
print(f"  Train : {len(df_train):>6,} rows  ({len(df_train)/len(df_clean)*100:.1f}%)")
print(f"  Val   : {len(df_val):>6,} rows  ({len(df_val)/len(df_clean)*100:.1f}%)")
print(f"  Test  : {len(df_test):>6,} rows  ({len(df_test)/len(df_clean)*100:.1f}%)")
print(f"  Total : {len(df_train)+len(df_val)+len(df_test):>6,} rows")

Split sizes (before scaling):
  Train :  8,488 rows  (80.0%)
  Val   :  1,061 rows  (10.0%)
  Test  :  1,061 rows  (10.0%)
  Total : 10,610 rows


## 2.6 — Fit `StandardScaler` on train targets only, transform val + test

In [6]:
scaler = StandardScaler()

# Fit ONLY on training targets — prevents data leakage
df_train = df_train.copy()
df_val   = df_val.copy()
df_test  = df_test.copy()

df_train[TARGET_COLS] = scaler.fit_transform(df_train[TARGET_COLS])
df_val[TARGET_COLS]   = scaler.transform(df_val[TARGET_COLS])
df_test[TARGET_COLS]  = scaler.transform(df_test[TARGET_COLS])

print("StandardScaler fitted on train split.")
print("\nScaler means (7 targets):")
for col, mean, std in zip(TARGET_COLS, scaler.mean_, scaler.scale_):
    print(f"  {col:<8s}  mean={mean:>9.4f}   std={std:>9.4f}")

print("\nPost-scaling train target stats (should be ~0 mean, ~1 std):")
print(df_train[TARGET_COLS].describe().loc[["mean", "std"]].round(4).to_string())

StandardScaler fitted on train split.

Scaler means (7 targets):
  PM2.5     mean= 132.3076   std= 129.5802
  PM10      mean= 139.1754   std= 106.6684
  O3        mean=  37.0166   std=  32.7284
  CO        mean= 107.1348   std= 119.8151
  SO2       mean=  13.1887   std=   9.9044
  NO2       mean=  31.8200   std=  34.1497
  AQI       mean= 166.4660   std= 109.0597

Post-scaling train target stats (should be ~0 mean, ~1 std):
       PM2.5    PM10      O3      CO     SO2     NO2     AQI
mean  0.0000  0.0000  0.0000 -0.0000 -0.0000 -0.0000 -0.0000
std   1.0001  1.0001  1.0001  1.0001  1.0001  1.0001  1.0001


## 2.7 — Save scaler and split CSVs

In [7]:
# Save scaler
scaler_path = ARTIFACTS_DIR / "scaler.pkl"
with open(scaler_path, "wb") as f:
    pickle.dump(scaler, f)
print(f"✔  Scaler saved  → {scaler_path}")

# Save split CSVs
train_path = DATA_DIR / "train.csv"
val_path   = DATA_DIR / "val.csv"
test_path  = DATA_DIR / "test.csv"

df_train.to_csv(train_path, index=False)
df_val.to_csv(val_path,     index=False)
df_test.to_csv(test_path,   index=False)

print(f"✔  Train CSV saved → {train_path}")
print(f"✔  Val   CSV saved → {val_path}")
print(f"✔  Test  CSV saved → {test_path}")

✔  Scaler saved  → C:\Users\Meraj\Desktop\PROJECTS\airsight\artifacts\scaler.pkl
✔  Train CSV saved → C:\Users\Meraj\Desktop\PROJECTS\airsight\data\train.csv
✔  Val   CSV saved → C:\Users\Meraj\Desktop\PROJECTS\airsight\data\val.csv
✔  Test  CSV saved → C:\Users\Meraj\Desktop\PROJECTS\airsight\data\test.csv


## 2.8 — Final row count verification

In [8]:
# Re-read from disk to confirm what was actually written
train_check = pd.read_csv(train_path)
val_check   = pd.read_csv(val_path)
test_check  = pd.read_csv(test_path)

total = len(train_check) + len(val_check) + len(test_check)

print("=" * 45)
print("  FINAL SPLIT ROW COUNTS (read from disk)")
print("=" * 45)
print(f"  Train : {len(train_check):>6,} rows")
print(f"  Val   : {len(val_check):>6,} rows")
print(f"  Test  : {len(test_check):>6,} rows")
print("-" * 45)
print(f"  Total : {total:>6,} rows")
print("=" * 45)
print(f"\nColumns saved: {train_check.columns.tolist()}")
print(f"Scaler exists: {scaler_path.exists()}")
print("\n✔  Step 2 complete. Proceed to notebook 03_train_model.ipynb")

  FINAL SPLIT ROW COUNTS (read from disk)
  Train :  8,488 rows
  Val   :  1,061 rows
  Test  :  1,061 rows
---------------------------------------------
  Total : 10,610 rows

Columns saved: ['image_path', 'PM2.5', 'PM10', 'O3', 'CO', 'SO2', 'NO2', 'AQI']
Scaler exists: True

✔  Step 2 complete. Proceed to notebook 03_train_model.ipynb
